%md
# 01 · Setup & Governance (Unity Catalog)
**Goal:** create the "building" where all project data lives.

- **Catalog** `procintel` = the building
- **Schemas** = floors: `bronze` (raw) → `silver` (clean) → `gold` (business-ready) + `ml` + `ops`
- **Volume** `bronze.raw_files` = a governed folder for *files* (JSON, report text)

Every table gets a 3-level name: `catalog.schema.table`. Permissions, lineage and audit attach to these names.
All statements use `IF NOT EXISTS`, so the notebook is **idempotent**: safe to re-run.

In [0]:
-- =====================================================================
-- CELL 1: Create our own catalog (the "building" for all project data)
-- =====================================================================
-- A CATALOG is the top level of Unity Catalog's 3-level namespace:
--      catalog . schema . table      e.g.  procintel.silver.contracts
-- Why our own catalog instead of the default "workspace" one?
--   -> Everything for this project is in one place, easy to grant
--      access to, easy to find, and easy to delete when done.
-- IF NOT EXISTS -> if it already exists, do nothing instead of failing.
--   This makes the notebook IDEMPOTENT: running it twice = same result.
-- COMMENT -> a description that shows up in the Catalog UI (documentation).

CREATE CATALOG IF NOT EXISTS procintel
COMMENT 'World Bank procurement intelligence demo';

In [0]:
-- =====================================================================
-- CELL 2: Create the schemas (the "floors" of the building)
-- =====================================================================
-- MEDALLION ARCHITECTURE = the standard Databricks pattern:
--   BRONZE -> raw data exactly as downloaded (never edited)
--   SILVER -> cleaned, de-duplicated, correct data types
--   GOLD   -> joined, business-ready tables the agents & dashboards read
-- Plus two extra schemas for this project:
--   ML     -> trained models and the vector search index
--   OPS    -> our own logs: agent decisions, evaluation scores
-- Why separate schemas? PERMISSIONS. Example: analysts can read GOLD
--   but never touch BRONZE, with one GRANT per schema.
-- Why keep BRONZE raw? If our cleaning code has a bug, we fix it and
--   re-run from bronze, with no need to download everything again.

USE CATALOG procintel;   -- "work inside this building from now on",
                         -- so we can write "bronze" instead of "procintel.bronze"

CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Raw data exactly as downloaded';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Cleaned and typed tables';
CREATE SCHEMA IF NOT EXISTS gold   COMMENT 'Joined, business-ready tables and features';
CREATE SCHEMA IF NOT EXISTS ml     COMMENT 'Models and vector indexes';
CREATE SCHEMA IF NOT EXISTS ops    COMMENT 'Agent logs, evaluation results, audit views';

In [0]:
-- =====================================================================
-- CELL 3: Create a VOLUME (a governed folder for FILES)
-- =====================================================================
-- TABLES hold rows and columns (structured data).
-- VOLUMES hold files: JSON downloads, PDFs, report text (unstructured data).
-- Both are governed by Unity Catalog -> same permissions, same audit log.
-- This is the LAKEHOUSE idea: structured + unstructured data in ONE
--   governed place, instead of a database here and a file share there.
-- The volume's file path will be:  /Volumes/procintel/bronze/raw_files/

CREATE VOLUME IF NOT EXISTS procintel.bronze.raw_files
COMMENT 'Raw JSON downloads and report text files';

In [0]:
-- =====================================================================
-- CELL 4: Verify -> list all schemas in our catalog
-- =====================================================================
-- Expected: bronze, silver, gold, ml, ops
-- (+ "default" and "information_schema", which Databricks creates
--  automatically in every catalog: information_schema = metadata
--  about every table and column)

SHOW SCHEMAS IN procintel;

In [0]:
-- =====================================================================
-- CELL 5: PERMISSIONS -> who can do what (the "G" in governance)
-- =====================================================================
-- Unity Catalog permissions follow the 3-level hierarchy. To read a
-- table, a user needs ALL THREE:
--   USE CATALOG  -> may "enter the building"
--   USE SCHEMA   -> may "go to that floor"
--   SELECT       -> may "read the rooms (tables) on that floor"
-- `account users` = a built-in group containing every user in the account.
-- PRINCIPLE OF LEAST PRIVILEGE: give only what is needed.
--   Consumers (analysts, the chatbot's users) read GOLD only.
--   They get NO access to BRONZE (raw, unvalidated) or OPS (logs).
-- In a company you'd grant to groups like `procurement_analysts`,
-- never to individual people, so joiners/leavers are managed in one place.

GRANT USE CATALOG ON CATALOG procintel TO `account users`;
GRANT USE SCHEMA, SELECT ON SCHEMA procintel.gold TO `account users`;

-- Verify: list who has which permission on the gold schema
SHOW GRANTS ON SCHEMA procintel.gold;

In [0]:
-- =====================================================================
-- CELL 6: TAGS -> labels that describe the data (searchable metadata)
-- =====================================================================
-- Tags are key = value labels attached to catalogs, schemas, tables or
-- even single columns. Uses:
--   * DISCOVERY  -> search the catalog for "layer = gold"
--   * GOVERNANCE -> e.g. tag columns with personal data as pii = true,
--                   then build policies on that tag
--   * AUDIT      -> show reviewers where the data came from (source)
-- Our data is PUBLIC World Bank data, so we record that explicitly.

ALTER SCHEMA procintel.bronze SET TAGS ('layer' = 'bronze', 'source' = 'world_bank_open_data', 'sensitivity' = 'public');
ALTER SCHEMA procintel.silver SET TAGS ('layer' = 'silver', 'source' = 'world_bank_open_data', 'sensitivity' = 'public');
ALTER SCHEMA procintel.gold   SET TAGS ('layer' = 'gold',   'source' = 'world_bank_open_data', 'sensitivity' = 'public');
ALTER SCHEMA procintel.ml     SET TAGS ('layer' = 'ml');
ALTER SCHEMA procintel.ops    SET TAGS ('layer' = 'ops',    'sensitivity' = 'internal');

In [0]:
-- =====================================================================
-- APP SERVICE PRINCIPAL -> least privilege: only what the chatbot needs
-- =====================================================================
-- Replace <APP_SP_ID> with the app's service principal application ID (a UUID).
-- It can READ Gold, SEARCH the index, and APPEND to the log. Nothing else:
-- no Bronze, no Silver, no writing to data tables.
GRANT USE CATALOG ON CATALOG procintel                    TO `35f9e74a-ad61-4fe7-9f9c-7de8aa252ac3`;
GRANT USE SCHEMA, SELECT ON SCHEMA procintel.gold         TO `35f9e74a-ad61-4fe7-9f9c-7de8aa252ac3`;
GRANT USE SCHEMA ON SCHEMA procintel.ml                   TO `35f9e74a-ad61-4fe7-9f9c-7de8aa252ac3`;
GRANT SELECT ON TABLE procintel.ml.rag_focus_index        TO `35f9e74a-ad61-4fe7-9f9c-7de8aa252ac3`;
GRANT USE SCHEMA ON SCHEMA procintel.ops                  TO `35f9e74a-ad61-4fe7-9f9c-7de8aa252ac3`;
GRANT SELECT, MODIFY ON TABLE procintel.ops.agent_requests TO `35f9e74a-ad61-4fe7-9f9c-7de8aa252ac3`;